# Welcome to Week 2!

## Frontier Model APIs

In Week 1, we used multiple Frontier LLMs through their Chat UI, and we connected with the OpenAI's API.

Today we'll connect with them through their APIs..

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Important Note - Please read me</h2>
            <span style="color:#900;">I'm continually improving these labs, adding more examples and exercises.
            At the start of each week, it's worth checking you have the latest code.<br/>
            First do a git pull and merge your changes as needed</a>. Check out the GitHub guide for instructions. Any problems? Try asking ChatGPT to clarify how to merge - or contact me!<br/>
            </span>
        </td>
    </tr>
</table>
<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder about the resources page</h2>
            <span style="color:#f71;">Here's a link to resources for the course. This includes links to all the slides.<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            Please keep this bookmarked, and I'll continue to add more useful links there over time.
            </span>
        </td>
    </tr>
</table>

In [ ]:
import os
from dotenv import load_dotenv
from openai import AzureOpenAI, OpenAI
from IPython.display import Markdown, display, update_display
from App.config import azure_endpoint, api_version, headers

In [ ]:
load_dotenv(override=True)

api_key = os.getenv('cd_api_key')
gemini_api_key = os.getenv('GOOGLE_API_KEY')

if api_key and api_key.startswith('e4'):
    print('API key found, and looks good so far')
else:
    print('API key issue, please check')    

if gemini_api_key:
    print(f"Google API Key exists and begins {gemini_api_key[:2]}")
else:
    print("Google API Key not set")


In [ ]:
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(api_key=gemini_api_key, base_url=gemini_url)

In [ ]:
def call_openai(model, messages, **kwargs):
    openai = AzureOpenAI(
        api_key=api_key,
        azure_endpoint=azure_endpoint,
        api_version=api_version
    )

    response = openai.chat.completions.create(
        messages=messages,
        model=model,
        extra_headers=headers,
        **kwargs
    )

    if (kwargs.get('stream')):
        streaming = response

        res = ''
        handle_display = display(Markdown(''), display_id=True)
        
        for chunk in streaming:
            if not chunk.choices:
                continue
            delta = chunk.choices[0].delta 

            res += delta.content or ''
            update_display(Markdown(res), display_id=handle_display.display_id)
    else:
        return response.choices[0].message.content


In [ ]:
tell_a_joke = [
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

model = 'gpt-5-nano'

call_openai(model, tell_a_joke, stream=True)

## Training vs Inference time scaling

In [ ]:
easy_puzzle = [
    {"role": "user", "content": 
    "You toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only."}
]

In [ ]:
result = call_openai(model, easy_puzzle, reasoning_effort='low')
display(Markdown(result))

## Testing out with hard problems

In [ ]:
hard = """
On a bookshelf, two volumes of Pushkin stand side by side: the first and the second.
The pages of each volume together have a thickness of 2 cm, and each cover is 2 mm thick.
A worm gnawed (perpendicular to the pages) from the first page of the first volume to the last page of the second volume.
What distance did it gnaw through?
Do not respond in code blocks. Think Step by Step
"""
hard_puzzle = [
    {"role": "user", "content": hard}
]

In [ ]:
model = 'gpt-5-mini'
result = call_openai(model, hard_puzzle, reasoning_effort='low')
display(Markdown(result))

## A spicy challenge to test the competitive spirit

In [ ]:
dilemma_prompt = """
You and a partner are contestants on a game show. You're each taken to separate rooms and given a choice:
Cooperate: Choose "Share" — if both of you choose this, you each win $1,000.
Defect: Choose "Steal" — if one steals and the other shares, the stealer gets $2,000 and the sharer gets nothing.
If both steal, you both get nothing.
Do you choose to Steal or Share? Pick one.
"""

dilemma = [
    {"role": "user", "content": dilemma_prompt},
]

In [ ]:
model = 'gpt-5-mini'
result = call_openai(model, dilemma, reasoning_effort='low')
display(Markdown(result))

## Going local

Just use the OpenAI library pointed to localhost:11434/v1

In [ ]:
import requests

requests.get('http://localhost:11434/').content

In [ ]:
!ollama pull llama3.2:1b

In [ ]:
ollama_url = "http://localhost:11434/v1"

In [ ]:
def call_ollama(messages, **kwargs):
    ollama = OpenAI(api_key='ollama', base_url=ollama_url)
    response = ollama.chat.completions.create(
        messages=messages,
        **kwargs
    )

    return response.choices[0].message.content

In [ ]:
result = call_ollama(messages=dilemma, model='llama3.2:1b')
display(Markdown(result))

### Gemini Endpoint Library

In [ ]:
from google import genai

In [ ]:
gemini = genai.Client()

def call_gemini(model):
    response = gemini.models.generate_content(
        model=model,
        contents="Describe the color Blue to someone who's never been able to see in 1 sentence"
    )
    print(f'Below is the response from the model {model}')
    print(response.text)
    return 

In [ ]:
call_gemini('gemini-3.1-flash-lite')

## Routers and Abtraction Layers

Starting with the wonderful OpenRouter.ai - it can connect to all the models above!

Visit openrouter.ai and browse the models.

Refer day 1 notebook

## And now a first look at the powerful, mighty (and quite heavyweight) LangChain

In [ ]:
## Below is just the code, but it won't work, since I don't have money for OpenAI API :/

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-5-nano')
response = llm.invoke(tell_a_joke)

display(Markdown(response))

In [ ]:
from langchain_google_genai import GoogleGenerativeAI

llm = GoogleGenerativeAI(model="gemini-2.5-flash-lite")
response = llm.invoke(tell_a_joke)

display(Markdown(response))

## Finally - my personal fave - the wonderfully lightweight LiteLLM

In [ ]:
from litellm import completion